In [4]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI

# 1. Cargar las variables del archivo .env
load_dotenv()

# 2. Obtener la clave desde las variables de entorno
api_key = os.getenv("FORGE_API_KEY")

if not api_key:
    raise ValueError("No se encontró FORGE_API_KEY en el archivo .env")

# 3. Configurar el cliente (ignorando la verificación SSL por el proxy corporativo)
client = OpenAI(
    api_key=api_key, 
    base_url="https://forge.plainconcepts.com/v1",
    http_client=httpx.Client(verify=False)
)

# 4. Hacer la petición al modelo para Text-to-SQL
response = client.chat.completions.create(
    model="glm-5-2",
    messages=[
        {"role": "system", "content": "You are a data engineering assistant. Translate natural language questions into valid SQL queries."},
        {"role": "user", "content": "Muéstrame el total de ventas agrupado por región para el último trimestre."}
    ],
    max_tokens=4096
)

print(response.choices[0].message.content)

Aquí tienes la consulta SQL. Asumo que tienes una tabla llamada `ventas` con columnas para la región, el monto de la venta y la fecha.

```sql
SELECT 
    region, 
    SUM(monto_venta) AS total_ventas
FROM 
    ventas
WHERE 
    fecha_venta >= DATE_TRUNC('quarter', CURRENT_DATE) - INTERVAL '3 months'
    AND fecha_venta < DATE_TRUNC('quarter', CURRENT_DATE)
GROUP BY 
    region
ORDER BY 
    total_ventas DESC;
```

### Notas sobre la consulta:
* **`DATE_TRUNC('quarter', CURRENT_DATE)`**: Esta función (estándar en PostgreSQL y otros motores) obtiene el primer día del trimestre actual.
* **Restar `INTERVAL '3 months'`**: Nos retrocede al primer día del trimestre anterior (el "último trimestre" completo). 
* **`WHERE`**: Asegura que solo se incluyan las fechas del trimestre calendario inmediatamente anterior. *(Nota: Si por "último trimestre" te refieres a los últimos 3 meses desde hoy en adelante, puedes cambiar la condición del `WHERE` a simplemente `WHERE fecha_venta >= CURRENT_DATE - 